# Pull deps

In [1]:
import sys
!{sys.executable} -m pip install torchaudio torchcodec datasets huggingface_hub scipy librosa "numpy<2.3"


  Using cached torchaudio-2.10.0-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
  Using cached torchcodec-0.10.0-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached datasets-4.6.1-py3-none-any.whl.metadata (19 kB)
  Using cached librosa-0.11.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached numpy-2.2.6-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
  Using cached pyarrow-23.0.1-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (3.1 kB)
  Using cached dill-0.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached pandas-3.0.1-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached xxhash-3.6.0-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (13 kB)
  Using cached multiprocess-0.70.18-py313-none-any.whl.metadata (7.2 kB)
  Using cached aiohttp-3.13.3-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (8.1 kB)
  Using cach

# Setup HF's storage manually (I need a separated HDD)

In [3]:
from pathlib import Path

import os


In [4]:
CACHE_DIR = Path("./.cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)


In [5]:
# Set root cache for all HF libs (datasets, hub, transformers, etc.)
_p = str(CACHE_DIR.resolve())
os.environ["HF_HOME"] = _p

print("Set HF home dir to", _p)


Set HF home dir to /home/jovyan/datasets/.cache


# Setting up

In [6]:
from subprocess import Popen, PIPE, CalledProcessError, run
from huggingface_hub import HfApi, list_repo_files
from datasets import Dataset, Audio, ClassLabel, load_dataset, concatenate_datasets, Value

from datasets.dataset_dict import DatasetDict
from pathlib import Path
from IPython.display import Audio as IPA, display
from concurrent.futures import ThreadPoolExecutor, as_completed
import datasets

import gc
import os
import sys
import glob
import librosa
import datasets
import numpy as np
import scipy.io as sio
import pandas as pd


/opt/conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
REPO_ID = "Hibou-Foundation/datian"
LOCAL_DIR = Path("./prepared_dataset")
REPO_TYPE = "dataset"


In [8]:
#The following DSs rely on these values
#geronimobasso/drone-audio-detection-samples
#yehiellevi/dataset-balanced-n-weighted-final
#Mixed datasets also follow this convention (at least the ones we currently have)
CLASSES_INV = {0:"other", 1: "drone"}
CLASSES = {"other": 0, "drone": 1}


In [9]:
dl_dir = Path("./downloaded_datasets")
local_tmp_dir = Path("./staging")


In [10]:
dl_dir.mkdir(parents=True, exist_ok=True)
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
local_tmp_dir.mkdir(parents=True, exist_ok=True)


In [11]:
def easy_display_audio(audio, sample_rate):
    if np.max(np.abs(audio)) > 1:
        audio = audio / np.max(np.abs(audio))

    # Display the audio player
    display(IPA(data=audio, rate=int(sample_rate.item())))


# Sources

In [8]:
download_args = [
    ["--insecure", "https://lambda-iot.uniud.it/UAV_DeepAcousticLocalizationRecognition_Datasets/Scenario%202%20Dataset.zip", "--output", f"{dl_dir}/scenario2.zip"],
    ["--insecure", "https://lambda-iot.uniud.it/UAV_DeepAcousticLocalizationRecognition_Datasets/Scenario%201%20Dataset.zip", "--output", f"{dl_dir}/scenario1.zip"],
    ["https://github.com/DroneDetectionThesis/Drone-detection-dataset/archive/refs/heads/master.zip", "--output", f"{dl_dir}/DroneDetectionThesis.Drone-detection-dataset.zip"],
    ["https://uc90e6a3b40e150d234227daf272.dl.dropboxusercontent.com/cd/0/get/C7eK5DuGCpRv4uV9pfWTsFUjgjNtBXCZpqBEFPVxcwY-jwBmfseQgQotPOZ3iBQQfdKJXZSjhkXlQVL5PclGCNhR7Sd3uxEBg9IeuD5RoBWJY7wH92mlASk-I-Y8Xsyynx18jNUG7gU7QP7XI5THb_zm/file?_download_id=1637074187773826591426354030248830796317808430536170451142051367&_log_download_success=1&_notify_domain=www.dropbox.com&dl=1", "--output", f"{dl_dir}/aira-uas.tar.gz"],
    ["https://zenodo.org/records/15391924/files/Microphone_array.zip?download=1", "--output", f"{dl_dir}/UaVirBASE.zip"],
    ["https://www.kaggle.com/api/v1/datasets/download/yehiellevi/dataset-balanced-n-weighted-final", "--output", f"{dl_dir}/yehiellevi.dataset-balanced-n-weighted-final.zip"],
    ["https://data.nasa.gov/docs/datasets/rfk401li/small_uav_acoustics.zip", "--output", f"{dl_dir}/nasa.small_uav_flyover_acoustics.zip"],
]

file_names = [
    "scenario2.zip",
    "scenario1.zip",
    "UaVirBASE.zip",
    "DroneDetectionThesis.Drone-detection-dataset.zip",
    "aira-uas.tar.gz",
    "nasa.small_uav_flyover_acoustics.zip",
    "yehiellevi.dataset-balanced-n-weighted-final.zip",
]


In [9]:
hf_mixed_sources = [
    "geronimobasso/drone-audio-detection-samples",
]

# ("ahlab-drone-project/DroneAudioSet", ['drone-only', 'drone-with-source', 'ground-truth', 'source-only'])
# drone-with-source has been selected because of its noisy mixtures.

hf_drone_sources = [
    ("ahlab-drone-project/DroneAudioSet", ["drone-with-source"]),
]

#    "sps44/fsdnoisy18k",
hf_other_sources = [
    "agkphysics/AudioSet",
    "FluidInference/musan",
]


# Download DS archives

In [13]:
last_dl = 0


In [14]:
def run_command(cmd, dry=False):
    if dry:
        print(" ".join(cmd))
        return None

    with Popen(cmd, stdout=PIPE, bufsize=1, universal_newlines=True) as p:
        for line in p.stdout:
            print(line, end='') # process line here

        if p.returncode != 0 and p.returncode is not None:
            raise CalledProcessError(p.returncode, p.args)

        if p.returncode is not None:
            print("Exited with code", p.returncode)


In [26]:
def unpack_archives(files, dry=True):
    for f in files:
        print("Unpacking", f)
        if f.endswith(".zip"):
            run_command(["unzip", str(dl_dir / f), "-d", f"{dl_dir}/{f[:-4]}"], dry)
        elif f.endswith(".tar.gz"):
            run_command(["mkdir", "-p", f"{dl_dir}/{f[:-7]}"], dry)
            run_command(["tar", "-zxvf", str(dl_dir / f), "-C", f"{dl_dir}/{f[:-7]}", "--strip-components=1"], dry)


In [27]:
for i in range(last_dl, len(download_args)):
    run_command(["curl"] + download_args[i])
    last_dl = i


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0


In [28]:
unpack_archives(file_names[2:], dry=False)


Unpacking UaVirBASE.zip
Archive:  downloaded_datasets/UaVirBASE.zip


replace downloaded_datasets/UaVirBASE/Microphone_array/20241115_093128/label.json? [y]es, [n]o, [A]ll, [N]one, [r]ename:  NULL
(EOF or read error, treating as "[N]one" ...)


Unpacking DroneDetectionThesis.Drone-detection-dataset.zip
Archive:  downloaded_datasets/DroneDetectionThesis.Drone-detection-dataset.zip
Unpacking aira-uas.tar.gz
Unpacking nasa.small_uav_flyover_acoustics.zip
Archive:  downloaded_datasets/nasa.small_uav_flyover_acoustics.zip
Unpacking yehiellevi.dataset-balanced-n-weighted-final.zip
Archive:  downloaded_datasets/yehiellevi.dataset-balanced-n-weighted-final.zip
   creating: downloaded_datasets/yehiellevi.dataset-balanced-n-weighted-final/yehiellevi.dataset-balanced-n-weighted-final/
   creating: downloaded_datasets/yehiellevi.dataset-balanced-n-weighted-final/yehiellevi.dataset-balanced-n-weighted-final/fold10/
  inflating: downloaded_datasets/yehiellevi.dataset-balanced-n-weighted-final/yehiellevi.dataset-balanced-n-weighted-final/fold10/3-180256-A-02___1.wav  
  inflating: downloaded_datasets/yehiellevi.dataset-balanced-n-weighted-final/yehiellevi.dataset-balanced-n-weighted-final/fold10/4-172736-A-363___1.wav  
  inflating: downloa

  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of downloaded_datasets/DroneDetectionThesis.Drone-detection-dataset.zip or
        downloaded_datasets/DroneDetectionThesis.Drone-detection-dataset.zip.zip, and cannot find downloaded_datasets/DroneDetectionThesis.Drone-detection-dataset.zip.ZIP, period.

gzip: stdin: not in gzip format
tar: Child returned status 1
tar: Error is not recoverable: exiting now
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of downloaded_datasets/nasa.small_uav_fl

  inflating: downloaded_datasets/yehiellevi.dataset-balanced-n-weighted-final/yehiellevi.dataset-balanced-n-weighted-final/fold10/59277-0-0-2___1.wav  
  inflating: downloaded_datasets/yehiellevi.dataset-balanced-n-weighted-final/yehiellevi.dataset-balanced-n-weighted-final/fold10/2-87799-A-243___1.wav  
  inflating: downloaded_datasets/yehiellevi.dataset-balanced-n-weighted-final/yehiellevi.dataset-balanced-n-weighted-final/fold10/u3___50.wav  
  inflating: downloaded_datasets/yehiellevi.dataset-balanced-n-weighted-final/yehiellevi.dataset-balanced-n-weighted-final/fold10/3-70962-A-41___1.wav  
  inflating: downloaded_datasets/yehiellevi.dataset-balanced-n-weighted-final/yehiellevi.dataset-balanced-n-weighted-final/fold10/4-184235-A-283___1.wav  
  inflating: downloaded_datasets/yehiellevi.dataset-balanced-n-weighted-final/yehiellevi.dataset-balanced-n-weighted-final/fold10/10m (5)___7.wav  
  inflating: downloaded_datasets/yehiellevi.dataset-balanced-n-weighted-final/yehiellevi.datas

# Pull in the HF datasets

In [22]:
hf_other = [load_dataset(ds) for ds in hf_other_sources]


Repo card metadata block was not found. Setting CardData to empty.


In [14]:
hf_mixed = [load_dataset(ds) for ds in hf_mixed_sources]


In [15]:
hf_drone = [load_dataset(ds) if ds is str else [load_dataset(ds[0], conf) for conf in ds[1]] for ds in hf_drone_sources]


# Process the downloaded archives

In [59]:
def gen_aira_uas():
    data_root = f"{dl_dir}/aira-uas/"

    protos = [f"Protocol{i}" for i in range(1, 4)]
    infos = {"Protocol1": ([i for i in range(4, 16)], 18), "Protocol2": ([18], 3), "Protocol3": ([21], 3)}
    
    drones = []
    others = []
    elements = []
    
    j = 0
    for i in range(1, 4):
        subp = f"Protocol{i}"
        for k in range(infos[subp][1]):
            path = data_root + subp + f"/Recording{j}"
            if j in infos[subp][0]:
                drones += glob.glob(path + "/*.wav")
            else:
                others += glob.glob(path + "/*.wav")
            j += 1

    for f, label in [(f, CLASSES["drone"]) for f in drones] + [(f, CLASSES["other"]) for f in others]:
        data, sr = librosa.load(f, sr=None, mono=False)
        cast = np.array(data, dtype=np.float32)
        if cast.ndim > 1:
            for i in range(cast.shape[0]):
                elements.append({"audio": {"array": cast[i], "sampling_rate": sr}, "label": label, "src": "aira"})
        else:
            elements.append({"audio": {"array": cast, "sampling_rate": sr}, "label": label, "src": "aira"})

    return Dataset.from_list(elements).cast_column("audio", Audio(decode=True))


In [60]:
def gen_ddd():
    data_root = f"{dl_dir}/DroneDetectionThesis.Drone-detection-dataset/Drone-detection-dataset-master/Data/Audio/"

    drones = glob.glob(data_root + "DRONE_*.wav")
    others = glob.glob(data_root + "BACKGROUND_*.wav") + glob.glob(data_root + "HELICOPTER_*.wav") + glob.glob(data_root + "DRONE_*.wav")

    elements = []
    
    for f in drones:
        data, sr = librosa.load(f, sr=None, mono=False)
        cast = np.array(data, dtype=np.float32)
        if cast.ndim > 1:
            for i in range(cast.shape[0]):
                elements.append({"audio": {"array": cast[i], "sampling_rate": sr}, "label": CLASSES["drone"]})
        else:
            elements.append({"audio": {"array": cast, "sampling_rate": sr}, "label": CLASSES["drone"]})

    for f in others:
        data, sr = librosa.load(f, sr=None, mono=False)
        cast = np.array(data, dtype=np.float32)
        if cast.ndim > 1:
            for i in range(cast.shape[0]):
                elements.append({"audio": {"array": cast[i], "sampling_rate": sr}, "label": CLASSES["other"], "src": "ddd"})
        else:
            elements.append({"audio": {"array": cast, "sampling_rate": sr}, "label": CLASSES["other"], "src": "ddd"})

    return Dataset.from_list(elements).cast_column("audio", Audio(decode=True))


In [61]:
def gen_dalrd():
    data_root_1 = f"{dl_dir}/scenario1/Scenario 1 Dataset/"
    data_root_2 = f"{dl_dir}/scenario2/Scenario 2 Dataset/"
    
    drones = glob.glob(data_root_1 + "*.wav", recursive=True) + glob.glob(data_root_2 + "*.wav", recursive=True)
    
    elements = []
    
    for f in drones:
        data, sr = librosa.load(f, sr=None, mono=False)
        cast = np.array(data, dtype=np.float32)
        # Check if multiple channels. If yes, split it.
        if cast.ndim > 1:
            for i in range(cast.shape[0]):
                elements.append({"audio": {"array": cast[i], "sampling_rate": sr}, "label": CLASSES["drone"], "src": "dalrd"})
        else:
            elements.append({"audio": {"array": cast, "sampling_rate": sr}, "label": CLASSES["drone"], "src": "dalrd"})
    
    return Dataset.from_list(elements).cast_column("audio", Audio(decode=True))


In [62]:
def gen_uavirbase():
    data_root = f"{dl_dir}UaVirBASE/Microphone_array/"
    drones = glob.glob(data_root + "*.wav", recursive=True)
    
    elements = []
    
    for f in drones:
        data, sr = librosa.load(f, sr=None, mono=False)
        cast = np.array(data, dtype=np.float32)
        # Check if multiple channels. If yes, split it.
        if cast.ndim > 1:
            for i in range(cast.shape[0]):
                elements.append({"audio": {"array": cast[i], "sampling_rate": sr}, "label": CLASSES["drone"], "src": "uavirbase"})
        else:
            elements.append({"audio": {"array": cast, "sampling_rate": sr}, "label": CLASSES["drone"], "src": "uavirbase"})
    
    return Dataset.from_list(elements).cast_column("audio", Audio()).cast_column("audio", Audio(decode=True))
    

In [63]:
def gen_yehiellevi(locally_zipped = False):
    data_root = f"{dl_dir}/yehiellevi.dataset-balanced-n-weighted-final/"
    if locally_zipped:
        data_root += "yehiellevi.dataset-balanced-n-weighted-final/"
    
    df = pd.read_csv(data_root + "audio_metadata_shuffled.csv", sep=",")
    
    elements = []
    
    for _, r in df.iterrows():
        path = f"{data_root}/fold{r['fold']}/{r['slice_file_name']}"
        start, end = r["start"], r["end"]
        data, sr = librosa.load(path, sr=None, offset=start, duration=end - start, mono=False)
        cast = np.array(data, dtype=np.float32)
        # Check if multiple channels. If yes, split it.
        if cast.ndim > 1:
            for i in range(cast.shape[0]):
                elements.append({"audio": {"array": cast[i], "sampling_rate": sr}, "label": r["classID"]})
        else:
            elements.append({"audio": {"array": cast, "sampling_rate": sr}, "label": r["classID"], "src": "yehiellevi"})
    
    return Dataset.from_list(elements).cast_column("audio", Audio(decode=True))
    

# NASA's DS is a bit more complicated than the others.

In [64]:
def load_mat(filepath: str) -> dict:
    """Load a .mat file, trying scipy first (v5), then h5py for v7.3."""
    try:
        mat = sio.loadmat(filepath, squeeze_me=True, struct_as_record=False)
        return mat
    except NotImplementedError:
        # v7.3 HDF5-based .mat files
        try:
            import h5py
        except ImportError:
            print("ERROR: h5py is required for MATLAB v7.3 files. Install with: pip install h5py")
            sys.exit(1)

        def hdf5_to_dict(h5obj):
            result = {}
            for key, val in h5obj.items():
                if isinstance(val, h5py.Dataset):
                    result[key] = val[()]
                elif isinstance(val, h5py.Group):
                    result[key] = hdf5_to_dict(val)
            return result

        import h5py
        with h5py.File(filepath, "r") as f:
            return hdf5_to_dict(f)


In [65]:
def get_field(struct, field: str):
    """Retrieve a field from either a scipy mat_struct or a plain dict."""
    if isinstance(struct, dict):
        return struct[field]
    return getattr(struct, field)


In [66]:
def extract_acoustic_data(mat: dict, mic_channel: int):
    """
    Extract acoustic pressure and UTC time from the 'acoustics' structure.

    Returns
    -------
    utc_time : np.ndarray  (n,)   seconds past midnight, UTC
    pressures : np.ndarray (n, num_mics)  incident pressures in Pascals
    selected_channel_pressure : np.ndarray (n,)  pressure for chosen mic
    sample_rate : float  Hz
    """
    acoustics = mat["acoustics"]

    utc_time   = np.asarray(get_field(acoustics, "utc_time"), dtype=float).ravel()
    pressures  = np.asarray(get_field(acoustics, "incident_pascals"), dtype=float)

    # Ensure 2-D: (n_samples, n_mics)
    if pressures.ndim == 1:
        pressures = pressures[:, np.newaxis]

    n_mics = pressures.shape[1]
    if mic_channel >= n_mics:
        raise ValueError(
            f"Requested channel {mic_channel} but data only has {n_mics} mic(s) "
            f"(0-based indexing)."
        )

    selected_pressure = pressures[:, mic_channel]

    # Sampling rate from mean time step
    dt          = np.mean(np.diff(utc_time))
    sample_rate = 1.0 / dt

    return utc_time, pressures, selected_pressure, sample_rate


In [67]:
def extract_vehicle_data(mat: dict):
    """Extract RTK and GPS vehicle position data."""
    vd = mat.get("vehicle_data")
    if vd is None:
        return None

    result = {}
    for field in ("rtk_utc_time", "rtk_ned_meters", "rtk_status",
                  "gps_utc_time", "gps_ned_meters"):
        try:
            result[field] = np.asarray(get_field(vd, field), dtype=float)
        except (AttributeError, KeyError):
            result[field] = None

    return result


In [68]:
def extract_met_data(mat: dict):
    """Extract meteorological data if available."""
    met = mat.get("met_data")
    if met is None:
        return None

    result = {}
    for field in ("temperature_celsius", "windspeed_knots", "wind_direction", "utc_time"):
        try:
            result[field] = np.asarray(get_field(met, field), dtype=float)
        except (AttributeError, KeyError):
            result[field] = None

    return result


In [69]:
def print_summary(utc_time, pressures, selected_pressure, sample_rate,
                  mic_channel, vehicle_data, met_data):
    """Print a human-readable summary to the console."""
    print("=" * 60)
    print("  NASA Small UAV Acoustic Data Extraction")
    print("=" * 60)

    duration = utc_time[-1] - utc_time[0]
    print(f"\n[ACOUSTICS]")
    print(f"  Sample rate      : {sample_rate:.2f} Hz")
    print(f"  Num samples      : {len(utc_time)}")
    print(f"  Duration         : {duration:.2f} s  ({duration/60:.2f} min)")
    print(f"  Num mic channels : {pressures.shape[1]}")
    print(f"  Selected channel : {mic_channel} (0-based)")
    print(f"  Pressure range   : [{selected_pressure.min():.6f}, {selected_pressure.max():.6f}] Pa")
    print(f"  Peak SPL (A-weighted ref 20µPa): "
          f"{20 * np.log10(np.sqrt(np.mean(selected_pressure**2)) / 20e-6):.1f} dB")

    if vehicle_data:
        print(f"\n[VEHICLE DATA]")
        rtk_t = vehicle_data.get("rtk_utc_time")
        gps_t = vehicle_data.get("gps_utc_time")
        ned_rtk = vehicle_data.get("rtk_ned_meters")
        ned_gps = vehicle_data.get("gps_ned_meters")
        if rtk_t is not None:
            print(f"  RTK samples      : {len(rtk_t.ravel())}")
        if gps_t is not None:
            print(f"  GPS samples      : {len(gps_t.ravel())}")
        if ned_rtk is not None and ned_rtk.ndim == 2:
            r = np.sqrt(np.sum(ned_rtk**2, axis=1))
            print(f"  RTK max distance : {r.max():.2f} m from origin")
        if ned_gps is not None and ned_gps.ndim == 2:
            r = np.sqrt(np.sum(ned_gps**2, axis=1))
            print(f"  GPS max distance : {r.max():.2f} m from origin")

    if met_data:
        print(f"\n[METEOROLOGICAL DATA]")
        temp = met_data.get("temperature_celsius")
        wind = met_data.get("windspeed_knots")
        if temp is not None:
            print(f"  Avg temperature  : {np.nanmean(temp):.1f} °C")
        if wind is not None:
            print(f"  Avg wind speed   : {np.nanmean(wind):.1f} kn")
    else:
        print("\n[METEOROLOGICAL DATA] Not available")

    print()


In [70]:
def extract_nasa_data(mat_file, vehicle: str=None, channel: int=None):
    # Resolve mic channel
    if channel is not None:
        mic_channel = channel
    elif vehicle is not None:
        mic_channel = VEHICLE_MIC_CHANNEL[vehicle]
    else:
        mic_channel = 0
        print(f"No --vehicle or --channel specified; defaulting to channel 0.")

    # Load
    print(f"Loading: {mat_file}")
    mat = load_mat(mat_file)

    # Extract
    utc_time, pressures, selected_pressure, sample_rate = extract_acoustic_data(
        mat, mic_channel
    )

    return pressures[:, mic_channel], int(sample_rate)


In [71]:
def gen_nasa():
    data_root = f"{dl_dir}/nasa.small_uav_flyover_acoustics/data/"
    files = glob.glob(data_root + "*.mat")

    drones = []

    for f in files:
        basename = os.path.basename(f)
        if basename.startswith("cub_") or basename.startswith("hex_"):
            channel = 0
        else:
            channel = 2
        data, sr = extract_nasa_data(f, channel=channel)
        cast = np.array(data, dtype=np.float32)
        # Check if multiple channels. If yes, split it.
        if cast.ndim > 1:
            for i in range(cast.shape[0]):
                drones.append({"array": cast[i], "sampling_rate": sr})
        else:
            drones.append({"array": cast, "sampling_rate": sr})

    elements = [{"audio": f, "label": CLASSES["drone"], "src": "nasa"} for f in drones]
    return Dataset.from_list(elements).cast_column("audio", Audio(decode=True))


# Load local DSs

# Save what we've got so far.

In [72]:
_listed = [gen_aira_uas(), gen_ddd(), gen_dalrd(), gen_uavirbase(), gen_yehiellevi(True), gen_nasa()]


/tmp/ipykernel_918/1681943840.py:13: UserWarning: PySoundFile failed. Trying audioread instead.
  data, sr = librosa.load(path, sr=None, offset=start, duration=end - start, mono=False)
/opt/conda/lib/python3.13/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_918/1681943840.py:13: UserWarning: PySoundFile failed. Trying audioread instead.
  data, sr = librosa.load(path, sr=None, offset=start, duration=end - start, mono=False)
/opt/conda/lib/python3.13/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


In [73]:
unified_locally = concatenate_datasets(_listed)


In [74]:
gc.collect()

54378

In [75]:
unified_locally.save_to_disk(str(local_tmp_dir) + "/local_gathering")


Saving the dataset (1/1 shards): 100%|██████████| 21391/21391 [00:00<00:00, 25041.83 examples/s]


In [76]:
gc.collect()


109

In [77]:
print(unified_locally.features)


{'audio': Audio(sampling_rate=None, decode=True, num_channels=None, stream_index=None), 'label': Value('int64'), 'src': Value('string')}


# Uniformise the drone DSs

In [16]:
def ds_split_channels(ds):
    elements = []

    for i in range(len(ds)):
        elem = ds[i]
        sr = elem["audio"]["sampling_rate"]
        audios = np.array(elem["audio"]["array"]).T

        elements += [{"audio": {"sampling_rate": sr, "array": chan}} for chan in audios]

    ds = Dataset.from_list(elements).cast_column("audio", Audio(decode=True))
    return ds.add_column("label", [CLASSES["drone"]] * len(ds))


In [17]:
def processs_trains(src, start=0):
    _elems = [(k, ds) for k, ds in hf_drone[0][0].items()]
    count = len(_elems)

    for i in range(start, count):
        k, ds = _elems[i]
        print(f"{i}/{count}")
        v = ds_split_channels(ds)
        v.save_to_disk(str(local_tmp_dir) + "/" + k)


In [24]:
def process_single_train(args):
    i, k, ds, count, local_tmp_dir = args
    print(f"{i}/{count}")
    v = ds_split_channels(ds)
    v.save_to_disk(str(local_tmp_dir) + "/" + k)
    return i, k

def processs_trains(src, start=0, num_threads=4):
    _elems = [(k, ds) for k, ds in src.items()]
    count = len(_elems)

    tasks = [
        (i, k, ds, count, local_tmp_dir)
        for i, (k, ds) in enumerate(_elems)
        if i >= start
    ]

    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        futures = {executor.submit(process_single_train, task): task for task in tasks}
        for future in as_completed(futures):
            try:
                i, k = future.result()
            except Exception as e:
                task = futures[future]
                print(f"Error processing {task[1]}: {e}")


IOStream.flush timed out


In [25]:
print(len([(k, ds) for k, ds in hf_drone[0][0].items()]))


252


In [26]:
print(hf_drone[0][0])


DatasetDict({
    train_001: Dataset({
        features: ['file_path', 'audio', 'data_type'],
        num_rows: 6
    })
    train_002: Dataset({
        features: ['file_path', 'audio', 'data_type'],
        num_rows: 6
    })
    train_003: Dataset({
        features: ['file_path', 'audio', 'data_type'],
        num_rows: 6
    })
    train_004: Dataset({
        features: ['file_path', 'audio', 'data_type'],
        num_rows: 6
    })
    train_005: Dataset({
        features: ['file_path', 'audio', 'data_type'],
        num_rows: 6
    })
    train_006: Dataset({
        features: ['file_path', 'audio', 'data_type'],
        num_rows: 6
    })
    train_007: Dataset({
        features: ['file_path', 'audio', 'data_type'],
        num_rows: 6
    })
    train_008: Dataset({
        features: ['file_path', 'audio', 'data_type'],
        num_rows: 6
    })
    train_009: Dataset({
        features: ['file_path', 'audio', 'data_type'],
        num_rows: 6
    })
    train_010: Dataset(

In [ ]:
processs_trains(hf_drone[0][0], num_threads=8)


0/252
1/252
2/252
3/252
4/252
5/252
6/252
7/252


Saving the dataset (1/1 shards): 100%|██████████| 34/34 [00:00<00:00, 63.27 examples/s] 


8/252


Saving the dataset (1/1 shards): 100%|██████████| 34/34 [00:00<00:00, 39.56 examples/s]


9/252


...

107/252


Saving the dataset (1/1 shards): 100%|██████████| 34/34 [00:01<00:00, 30.37 examples/s]


108/252


Saving the dataset (1/1 shards): 100%|██████████| 34/34 [00:01<00:00, 27.31 examples/s]


109/252


In [ ]:
gc.collect()


In [16]:
def load_splitted_drones_ds(args):
    i, k, count, local_tmp_dir = args
    print(f"{i}/{count}")
    return datasets.load_from_disk(str(local_tmp_dir) + "/" + k)

def load_splitted_drones(src, start=0, num_threads=6):
    _elems = [(k, ds) for k, ds in hf_drone[0][0].items() if k.startswith("train_")]
    count = len(_elems)

    tasks = [
        (i, k, count, local_tmp_dir)
        for i, (k, ds) in enumerate(_elems)
        if i >= start
    ]

    results = []
    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        futures = {executor.submit(load_splitted_drones_ds, task): task for task in tasks}
        for future in as_completed(futures):
            try:
                results.append(future.result())
            except Exception as e:
                task = futures[future]
                print(f"Error processing {task[1]}: {e}")

    return concatenate_datasets(results)


In [17]:
unified_hf_drones = load_splitted_drones(hf_drone[0][0])


0/252
1/252
2/252
3/252
4/252
5/252
6/252
7/252
8/252
9/252
10/252
11/252
12/252
13/252
14/252
15/252
16/252
17/252
18/252
19/252
20/252
21/252
22/252
23/252
24/252
25/252
26/252
27/252
28/252
29/252
30/252
31/252
32/252
33/252
34/252
35/252
36/252
37/252
38/252
39/252
40/252
41/252
42/252
43/252
44/252
45/252
46/252
47/252
48/252
49/252
50/252
51/252
52/252
53/252
54/252
55/252
56/252
57/252
58/252
59/252
60/252
61/252
62/252
63/252
64/252
65/252
66/252
67/252
68/252
69/252
70/252
71/252
72/252
73/252
74/252
75/252
76/252
77/252
78/252
79/252
80/252
81/252
82/252
83/252
84/252
85/252
86/252
87/252
88/252
89/252
90/252
91/252
92/252
93/252
94/252
95/252
96/252
97/252
98/252
99/252
100/252
101/252
102/252
103/252
104/252
105/252
106/252
107/252
108/252
109/252
110/252
111/252
112/252
113/252
114/252
115/252
116/252
117/252
118/252
119/252
120/252
121/252
122/252
123/252
124/252
125/252
126/252
127/252
128/252
129/252
130/252
131/252
132/252
133/252
134/252
135/252
136/252
137/252
138/25

In [18]:
unified_hf_drones.save_to_disk(str(local_tmp_dir) + "/foreign_drone_gathering")


Saving the dataset (63/63 shards): 100%|██████████| 8568/8568 [04:17<00:00, 33.31 examples/s]


In [19]:
gc.collect()


1222

# Uniformise the other DSs

In [23]:
def gen_others():
    hf_other_simplified = []

    for opt in hf_other:
        if str(type(opt)) == "<class 'datasets.Dataset'>":
            hf_other_simplified.append(opt)
        elif str(type(opt)) == "<class 'datasets.dataset_dict.DatasetDict'>":
            for k, ds in opt.items():
                cols = list(ds.features.keys())
                cols.remove("audio")
    
                hf_other_simplified.append(ds.remove_columns(cols).add_column("label", [CLASSES["other"]] * len(ds)))

    return concatenate_datasets(hf_other_simplified)


In [24]:
unified_hf_others = gen_others()


In [25]:
gc.collect()


36

In [26]:
unified_hf_others.save_to_disk(str(local_tmp_dir) + "/foreign_other_gathering")


Saving the dataset (100/100 shards): 100%|██████████| 36598/36598 [09:43<00:00, 62.70 examples/s]


In [27]:
gc.collect()


501

# Uniformise the mixed DSs

In [28]:
def simplify_audio(elem):
    return {"label": elem["label"], "audio": {"sampling_rate": elem["sampling_rate"], "array": np.array(elem["audio"])}}

def simplify_audio(batch):
    return {
        "label": batch["label"],
        "audio": [
            {"sampling_rate": sr, "array": np.array(audio)}
            for sr, audio in zip(batch["sampling_rate"], batch["audio"])
        ]
    }


In [31]:
def gen_mixed():
    hf_mixed_simplified = []

    for opt in hf_mixed:
        if str(type(opt)) == "<class 'datasets.Dataset'>":
            hf_mixed_simplified.append(opt)
        elif str(type(opt)) == "<class 'datasets.dataset_dict.DatasetDict'>":
            for k, ds in opt.items():
                hf_mixed_simplified.append(ds)

    hf_mixed_pretty = []
    for ds in hf_mixed_simplified:
        if "sampling_rate" in ds.features:
            # Complicated matters
            hf_mixed_pretty.append(ds.map(simplify_audio, batched=True).remove_columns(["sampling_rate"]).cast_column("audio", Audio(decode=True)).cast_column("label", Value("int64")))
        else:
            hf_mixed_pretty.append(ds.cast_column("label", Value("int64")))

    return concatenate_datasets(hf_mixed_pretty)
    

In [33]:
unified_hf_mixed = gen_mixed()


Casting the dataset: 100%|██████████| 180320/180320 [01:00<00:00, 2968.54 examples/s] 


In [34]:
gc.collect()

3254

In [35]:
unified_hf_mixed.save_to_disk(str(local_tmp_dir) + "/foreign_mixed_gathering")


Saving the dataset (15/15 shards): 100%|██████████| 180320/180320 [00:35<00:00, 5009.24 examples/s] 


In [36]:
gc.collect()


613

# Now, we can properly set everything up for the final stage :)

In [19]:
stages = ["foreign_mixed_gathering", "foreign_drone_gathering", "foreign_other_gathering", "local_gathering"]


In [20]:
stages_ds = [datasets.load_from_disk(str(local_tmp_dir) + "/" + stg) for stg in stages]


In [22]:
print([ds.features["label"] for ds in stages_ds])


[Value('int64'), Value('int64'), Value('int64'), Value('int64')]


In [28]:
print([ds.features["label"] for ds in stages_ds])


[Value('int64'), Value('int64'), Value('int64'), Value('int64')]


# Just ensure everythin is mono ;)

In [45]:
def is_valid_audio(example):
    """Filter out problematic audio before processing"""
    try:
        # Quick check if audio exists and has reasonable length
        _array = np.array(example["audio"]["array"])
        return len(_array) > 0 if isinstance(example["audio"], dict) else True
    except:
        return False


In [46]:
def split_channels(batch):
    new_audios = []
    new_labels = []

    for audio, label in zip(batch["audio"], batch["label"]):
        array = np.array(audio["array"])
        sr = audio["sampling_rate"]

        # Skip anything that is None or empty
        if array is None or array.size == 0:
            continue

        # Handle mono and multi-channel
        if array.ndim == 1 or array.shape[0] == 1:
            new_audios.append(audio)
            new_labels.append(label)
        else:
            for ch in range(array.shape[0]):
                new_audios.append({
                    "array": array[ch, :],
                    "sampling_rate": sr,
                })
                new_labels.append(label)

    return {"audio": new_audios, "label": new_labels}


In [32]:
label_feature = ClassLabel(names=["other", "drone"])


In [41]:
i = 3

In [47]:
for j in range(i, len(stages_ds)):
    i = j
    print("Processing DS", j)

    ds = stages_ds[j].filter(is_valid_audio, num_proc=4).map(split_channels, batched=True, num_proc=4, batch_size=8)
    ds.save_to_disk(str(local_tmp_dir) + "/" + stages[j] + ".mono")


Processing DS 3


Saving the dataset (1/1 shards): 100%|██████████| 13949/13949 [00:01<00:00, 11686.17 examples/s]


In [48]:
gc.collect()


5127

# Now we can save it all as one DS.

In [49]:
result_ds = [datasets.load_from_disk(str(local_tmp_dir) + "/" + stg + ".mono") for stg in stages]


In [50]:
output_ds = concatenate_datasets(stages_ds)


In [25]:
num_shards = 325


In [55]:
for i in range(num_shards):
    print(f"{i}/{num_shards}")
    shard = output_ds.shard(index=i, num_shards=num_shards, contiguous=True)
    shard.to_parquet(f"{LOCAL_DIR}/output/shard_{i:05d}.parquet")


0/325


Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00,  4.18ba/s]


1/325


Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00,  7.49ba/s]


2/325


...

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 32.84ba/s]


323/325


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 19.71ba/s]


324/325


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 30.72ba/s]


# Try to load it, too see quickly if everything's good.

In [14]:
output_ds = datasets.load_dataset("parquet", data_dir=f"{LOCAL_DIR}/output/")


In [15]:
gc.collect()


254

# We're never sure enough, we'll check that the final DS only contains mono.

In [19]:
invalids = []
not_mono = []


In [20]:
def is_mono_audio(audio_dict):
    return audio_dict["array"].ndim == 1  # 1D = mono, 2D = multi-channel


In [21]:
# Full check
all_mono = True
count = len(output_ds["train"])
j = 0
for example in output_ds["train"]:
    try:
        if not is_mono_audio(example["audio"]):
            all_mono = False
            not_mono.append(j)
    except:
        invalids.append(j)

    j += 1

print(f"All audio is mono: {all_mono}")


All audio is mono: True


In [22]:
print(invalids)
print(not_mono)


[225486, 225490, 225492, 225493, 225500, 225509, 225512, 225515, 225516, 225517, 225518, 225525, 225530, 225532, 225533, 225534, 225542, 225545, 225547, 225548, 225549, 225550, 225554, 225557, 225558, 225559, 225561, 225562, 225563, 225564, 225566, 225567, 225570, 225574, 225579, 225581, 225583, 225585, 225588, 225594, 225596, 225598, 225600, 225605, 225608, 225609, 225611, 225616, 225618, 225623, 225627, 225628, 225629, 225638, 225639, 225644, 225646, 225647, 225649, 225650, 225651, 225655, 225657, 225659, 225660, 225663, 225664, 225665, 225667, 225668, 225671, 225672, 225677, 225679, 225684, 225685, 225689, 225696, 225698, 225702, 225708, 225710, 225715, 225716, 225717, 225718, 225724, 225727, 225728, 225732, 225736, 225741, 225744, 225745, 225747, 225749, 225751, 225752, 225755, 225756, 225757, 225758, 225764, 225765, 225766, 225767, 225769, 225772, 225774, 225775, 225778, 225780, 225784, 225785, 225789, 225790, 225795, 225798, 225799, 225801, 225804, 225808, 225809, 225810, 225814,

In [23]:
valid_mask = lambda example, idx: idx not in invalids


In [24]:
clean_ds = output_ds["train"].filter(valid_mask, with_indices=True).remove_columns(["src"])


Filter: 100%|██████████| 246877/246877 [13:01<00:00, 316.09 examples/s]


In [27]:
for i in range(num_shards):
    print(f"{i}/{num_shards}")
    shard = clean_ds.shard(index=i, num_shards=num_shards, contiguous=True)
    shard.to_parquet(f"{LOCAL_DIR}/output_cleaned/shard_{i:05d}.parquet")


0/325


Creating parquet from Arrow format: 100%|██████████| 3/3 [00:00<00:00,  6.54ba/s]


1/325


Creating parquet from Arrow format: 100%|██████████| 3/3 [00:00<00:00, 13.79ba/s]


2/325


...

Creating parquet from Arrow format: 100%|██████████| 3/3 [00:00<00:00, 57.76ba/s]


323/325


Creating parquet from Arrow format: 100%|██████████| 3/3 [00:00<00:00, 61.26ba/s]


324/325


Creating parquet from Arrow format: 100%|██████████| 3/3 [00:00<00:00, 59.33ba/s]


# Clear the remote repo if required.

In [12]:
token = ""


In [29]:
api = HfApi(token=token)


In [32]:
def clear_repo_files(api: HfApi, repo_id: str, repo_type: str):
    existing_files = set(list_repo_files(repo_id=repo_id, repo_type=repo_type))
    count = len(existing_files)
    i = 0
    for path in existing_files:
        print(f"{i}/{count} {path}")
        api.delete_file(path, repo_id=repo_id, repo_type=repo_type)
        i += 1


In [33]:
clear_repo_files(api, REPO_ID, REPO_TYPE)


0/173 shard_00261.parquet


HfHubHTTPError: (Request ID: Root=1-69aabd88-0137fa652d004b8439787d70;cd062673-3bf6-4ee7-887a-981d5f490215)

429 Too Many Requests: you have reached your 'api' rate limit.
Retry after 124 seconds (986/1000 requests remaining in current 300s window).
Url: https://huggingface.co/api/datasets/Hibou-Foundation/datian/commit/main.
You have exceeded the rate limit for repository commits (128 per hour). You can retry this action in about 1 hour. To reduce the number of commits, you can upload entire folders at once using the Hub Python library: https://huggingface.co/docs/huggingface_hub/guides/upload#upload-a-folder or for large folders: https://huggingface.co/docs/huggingface_hub/guides/upload#upload-a-large-folder. To increase your rate limit for this action, you can upgrade to a paid plan at https://huggingface.co/pricing.

# If you don't have enough API requests, just remove it.

In [13]:
from huggingface_hub import delete_repo


In [14]:
delete_repo(token=token, repo_id=REPO_ID, repo_type=REPO_TYPE)


In [15]:
gc.collect()


849

# Now we can perform the upload.

In [15]:
from huggingface_hub import create_repo


In [16]:
token = ""


In [17]:
api = HfApi(token=token)


In [12]:
def upload_directory(api: HfApi, token: str, local_dir: str, repo_id: str, repo_type: str = "dataset"):
    """
    Uploads all files from `local_dir` to the Hugging Face Hub repository.
    Automatically skips already uploaded files (resumable).
    """

    create_repo(repo_id, repo_type=repo_type, token=token, exist_ok=True)

    local_dir = Path(local_dir)
    if not local_dir.exists():
        raise ValueError(f"Local directory does not exist: {local_dir}")

    print(f"📂 Scanning directory: {local_dir}")
    files_to_upload = [p for p in local_dir.rglob("*") if p.is_file()]
    print(f"Found {len(files_to_upload)} files to check.")

    # 🧠 Get list of already uploaded files
    print(f"🔍 Fetching existing files in repo: {repo_id}")
    existing_files = set(list_repo_files(repo_id=repo_id, repo_type=repo_type))
    print(f"Repo already has {len(existing_files)} files.")

    uploaded_count = 0
    skipped_count = 0

    for fpath in files_to_upload:
        # Normalize path in repo (relative to base directory)
        path_in_repo = str(fpath.relative_to(local_dir)).replace("\\", "/")

        if path_in_repo in existing_files:
            print(f"⏩ Skipping already uploaded: {path_in_repo}")
            skipped_count += 1
            continue

        try:
            print(f"⬆️ Uploading: {path_in_repo} ...")
            api.upload_file(
                path_or_fileobj=str(fpath),
                path_in_repo=path_in_repo,
                repo_id=repo_id,
                repo_type=repo_type,
            )
            uploaded_count += 1
            print(f"✅ Uploaded: {path_in_repo}")
        except Exception as e:
            print(f"❌ Error uploading {fpath}: {e}")
            print("Stopping — rerun this script to resume.")
            break

    print("\n✅ Upload complete.")
    print(f"Uploaded: {uploaded_count}, Skipped: {skipped_count}")


In [13]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from huggingface_hub import HfApi, create_repo, list_repo_files


def upload_directory_parrallel(api: HfApi, token: str, local_dir: str, repo_id: str, repo_type: str = "dataset", max_workers: int = 4):
    """
    Uploads all files from `local_dir` to the Hugging Face Hub repository.
    Automatically skips already uploaded files (resumable).
    Uploads up to 4 files in parallel.
    """

    create_repo(repo_id, repo_type=repo_type, token=token, exist_ok=True)

    local_dir = Path(local_dir)
    if not local_dir.exists():
        raise ValueError(f"Local directory does not exist: {local_dir}")

    print(f"📂 Scanning directory: {local_dir}")
    files_to_upload = [p for p in local_dir.rglob("*") if p.is_file()]
    print(f"Found {len(files_to_upload)} files to check.")

    # Get list of already uploaded files
    print(f"🔍 Fetching existing files in repo: {repo_id}")
    existing_files = set(list_repo_files(repo_id=repo_id, repo_type=repo_type))
    print(f"Repo already has {len(existing_files)} files.")

    skipped_count = 0
    upload_tasks = []

    for fpath in files_to_upload:
        path_in_repo = str(fpath.relative_to(local_dir)).replace("\\", "/")

        if path_in_repo in existing_files:
            print(f"⏩ Skipping already uploaded: {path_in_repo}")
            skipped_count += 1
            continue

        upload_tasks.append((fpath, path_in_repo))

    print(f"🚀 Uploading {len(upload_tasks)} files with {max_workers} parallel workers...\n")

    def upload_one(task):
        fpath, path_in_repo = task
        try:
            print(f"⬆️ Uploading: {path_in_repo}")
            api.upload_file(
                path_or_fileobj=str(fpath),
                path_in_repo=path_in_repo,
                repo_id=repo_id,
                repo_type=repo_type,
            )
            return ("ok", path_in_repo)
        except Exception as e:
            return ("error", path_in_repo, str(e))

    uploaded_count = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(upload_one, task) for task in upload_tasks]

        for future in as_completed(futures):
            result = future.result()

            if result[0] == "ok":
                uploaded_count += 1
                print(f"✅ Uploaded: {result[1]}")
            else:
                print(f"❌ Error uploading {result[1]}: {result[2]}")
                print("Stopping — rerun this script to resume.")
                break

    print("\n✅ Upload complete.")
    print(f"Uploaded: {uploaded_count}, Skipped: {skipped_count}")


In [18]:
upload_directory(api, token, str(LOCAL_DIR) + "/output_cleaned", REPO_ID, REPO_TYPE)


📂 Scanning directory: prepared_dataset/output_cleaned
Found 325 files to check.
🔍 Fetching existing files in repo: Hibou-Foundation/datian
Repo already has 11 files.
⏩ Skipping already uploaded: shard_00070.parquet
⏩ Skipping already uploaded: shard_00315.parquet
⬆️ Uploading: shard_00281.parquet ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  97%|█████████▋| 1.00GB / 1.03GB,  295MB/s  
Processing Files (0 / 1):  97%|█████████▋| 1.00GB / 1.03GB,  279MB/s  
Processing Files (0 / 1):  98%|█████████▊| 1.01GB / 1.03GB,  252MB/s  
Processing Files (0 / 1):  98%|█████████▊| 1.01GB / 1.03GB,  240MB/s  
Processing Files (0 / 1):  98%|█████████▊| 1.01GB / 1.03GB,  230MB/s  
Processing Files (0 / 1):  99%|█████████▉| 1.02GB / 1.03GB,  221MB/s  
Processing Files (0 / 1):  99%|█████████▉| 1.02GB / 1.03GB,  213MB/s  
Processing Files (0 / 1): 100%|█████████▉| 1.02GB / 1.03GB,  205MB/s  
Processing Files (0 / 1): 100%|█████████▉| 1.03GB / 1.03GB,  197MB/s  
Processing Files (0 / 1): 100%|█████████▉| 1.03GB / 1.03GB,  191MB/s  
Processing Files (0 / 1): 100%|█████████▉| 1.03GB / 1.03GB,  184MB/s  
Processing Files (1 / 1): 100%|██████████| 1.03GB / 1.03GB,  166MB/s  
Processing Files (1 / 1): 100%|██████████| 1.03GB / 1.03GB,  143MB/s  
New Data U

✅ Uploaded: shard_00281.parquet
⏩ Skipping already uploaded: shard_00002.parquet
⏩ Skipping already uploaded: shard_00241.parquet
⏩ Skipping already uploaded: shard_00251.parquet
⏩ Skipping already uploaded: shard_00119.parquet
⏩ Skipping already uploaded: shard_00148.parquet
⏩ Skipping already uploaded: shard_00113.parquet
⏩ Skipping already uploaded: shard_00172.parquet
⏩ Skipping already uploaded: shard_00232.parquet
⬆️ Uploading: shard_00269.parquet ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  96%|█████████▌|  977MB / 1.02GB,  302MB/s  
Processing Files (0 / 1):  96%|█████████▌|  979MB / 1.02GB,  227MB/s  
Processing Files (0 / 1):  96%|█████████▌|  980MB / 1.02GB,  182MB/s  
Processing Files (0 / 1):  96%|█████████▋|  982MB / 1.02GB,  147MB/s  
Processing Files (0 / 1):  97%|█████████▋|  983MB / 1.02GB,  114MB/s  
Processing Files (0 / 1):  97%|█████████▋|  985MB / 1.02GB, 95.1MB/s  
Processing Files (0 / 1):  97%|█████████▋|  986MB / 1.02GB, 55.9MB/s  
Processing Files (0 / 1):  97%|█████████▋|  987MB / 1.02GB, 18.9MB/s  
Processing Files (0 / 1):  97%|█████████▋|  987MB / 1.02GB, 13.3MB/s  
Processing Files (0 / 1):  97%|█████████▋|  989MB / 1.02GB, 1.15MB/s  
Processing Files (0 / 1):  97%|█████████▋|  990MB / 1.02GB, 1.15MB/s  
Processing Files (0 / 1):  97%|█████████▋|  991MB / 1.02GB, 1.10MB/s  
Processing Files (0 / 1):  97%|█████████▋|  993MB / 1.02GB, 1.10MB/s  
Processing

✅ Uploaded: shard_00269.parquet
⬆️ Uploading: shard_00109.parquet ...


...

✅ Uploaded: shard_00126.parquet
⬆️ Uploading: shard_00177.parquet ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  83%|████████▎ | 9.87MB / 11.9MB,   ???B/s  
Processing Files (0 / 1):  96%|█████████▌| 11.4MB / 11.9MB, 1.31MB/s  
Processing Files (1 / 1): 100%|██████████| 11.9MB / 11.9MB,  472kB/s  
Processing Files (1 / 1): 100%|██████████| 11.9MB / 11.9MB,  432kB/s  
New Data Upload: 100%|██████████| 5.75MB / 5.75MB,  432kB/s  


✅ Uploaded: shard_00177.parquet
⬆️ Uploading: shard_00246.parquet ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):   0%|          | 5.09MB / 2.62GB, 6.37MB/s  
Processing Files (0 / 1):   0%|          | 6.79MB / 2.62GB, 3.39MB/s  
Processing Files (0 / 1):   0%|          | 8.49MB / 2.62GB, 2.83MB/s  
Processing Files (0 / 1):   0%|          | 10.2MB / 2.62GB, 2.43MB/s  
Processing Files (0 / 1):   0%|          | 11.3MB / 2.62GB, 2.18MB/s  
Processing Files (0 / 1):   0%|          | 13.0MB / 2.62GB, 2.10MB/s  
Processing Files (0 / 1):   1%|          | 14.7MB / 2.62GB, 2.10MB/s  
Processing Files (0 / 1):   1%|          | 16.4MB / 2.62GB, 2.05MB/s  
Processing Files (0 / 1):   1%|          | 18.1MB / 2.62GB, 2.01MB/s  
Processing Files (0 / 1):   1%|          | 19.3MB / 2.62GB, 1.96MB/s  
Processing Files (0 / 1):   1%|          | 21.0MB / 2.62GB, 1.94MB/s  
Processing Files (0 / 1):   1%|          | 22.7MB / 2.62GB, 1.72MB/s  
Processing Files (0 / 1):   1%|          | 24.4MB / 2.62GB, 1.72MB/s  
Processing

KeyboardInterrupt: 